### Q1. Embedding a query

In [1]:
import numpy as np
from tqdm.auto import tqdm

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

2026-06-29 14:38:52.250175044 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
embedder = Embedder()

query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

print(v.shape)
print(v[0])

(384,)
-0.02058203437252893


### Q2. Cosine similarity

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
print(len(documents))
print(documents[0].keys())

72
dict_keys(['content', 'filename'])


In [6]:
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"

doc = next(d for d in documents if d["filename"] == target_file)

doc_vector = embedder.encode(doc["content"])

score = doc_vector.dot(v)

print(score)

0.36107027225589694


### 3. Chunking and search by hand

In [7]:
chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))
print(chunks[0].keys())

295
dict_keys(['start', 'content', 'filename'])


In [8]:
chunk_vectors = []

for chunk in tqdm(chunks):
    chunk_vector = embedder.encode(chunk["content"])
    chunk_vectors.append(chunk_vector)

X = np.array(chunk_vectors)

print(X.shape)

  0%|          | 0/295 [00:00<?, ?it/s]

(295, 384)


In [9]:
scores = X.dot(v)

best_idx = scores.argmax()
best_chunk = chunks[best_idx]

print(best_idx)
print(scores[best_idx])
print(best_chunk["filename"])

94
0.6489017718578813
02-vector-search/lessons/07-sqlitesearch-vector.md


### 4. Vector search with minsearch

In [ ]:
query_q4 = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(query_q4)

In [12]:
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [13]:
vector_results = vector_index.search(q4_vector, num_results=5)

for result in vector_results:
    print(result["filename"])

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


### Q5. Text search vs vector search

In [14]:
query_q5 = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(query_q5)

In [15]:
vector_results_q5 = vector_index.search(q5_vector, num_results=5)

print("Vector results:")
for result in vector_results_q5:
    print(result["filename"])

Vector results:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [16]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

text_results_q5 = text_index.search(query_q5, num_results=5)

print("Text results:")
for result in text_results_q5:
    print(result["filename"])

Text results:
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [17]:
vector_files = {r["filename"] for r in vector_results_q5}
text_files = {r["filename"] for r in text_results_q5}

print(vector_files - text_files)

{'02-vector-search/lessons/08-pgvector.md'}


### Q6. Hybrid search using RRF

In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

In [19]:
query_q6 = "How do I give the model access to tools?"

q6_vector = embedder.encode(query_q6)

vector_results_q6 = vector_index.search(q6_vector, num_results=5)
text_results_q6 = text_index.search(query_q6, num_results=5)

In [20]:
hybrid_results = rrf([vector_results_q6, text_results_q6])

for result in hybrid_results:
    print(result["filename"])

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
